In [ ]:
# @title **CELDA 3: APLICACIÓN DEL MÉTODO FIFO FÍSICO**

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("⚙️ CELDA 3: APLICACIÓN DEL MÉTODO FIFO FÍSICO (VERSIÓN CORREGIDA)")
print("="*80)
print("OBJETIVO: Ajustar inventario teórico para que sea IGUAL al inventario físico")
print("="*80)

# ===============================
# CONFIGURACIÓN
# ===============================
INPUT_DIR = "input"
OUTPUT_DIR = "output"

# ===============================
# 1. CARGAR TODOS LOS DATOS
# ===============================
print("\n📂 1. Cargando todos los datos...")

# 1.1 Datos base (NO MODIFICABLES)
df_salidas = pd.read_csv(f"{INPUT_DIR}/Salida_2025.csv")
df_inv_fisico = pd.read_csv(f"{INPUT_DIR}/Inventario_Fisico_2025.csv")

# 1.2 Datos a ajustar (MODIFICABLES)
df_ingresos = pd.read_csv(f"{INPUT_DIR}/Ingreso_2025.csv")
df_inv_2024 = pd.read_csv(f"{INPUT_DIR}/Inventario_2024.csv")

# 1.3 Datos completos (con información real)
df_errores_ingresos = pd.read_csv(f"{INPUT_DIR}/Log_Errores_Ingresos.csv")

# 1.4 Lista maestra y comparación
lista_maestra = pd.read_csv("Lista Maestra de Residuos.csv")
df_comparacion = pd.read_csv(f"{INPUT_DIR}/Comparacion_Inventarios.csv")

print(f"   ✅ Cargados {len(df_ingresos)} ingresos y {len(df_salidas)} salidas")
print(f"   ✅ Inventario físico 2025: {len(df_inv_fisico)} residuos")
print(f"   ✅ Diferencias identificadas: {len(df_comparacion)} residuos")

# ===============================
# 2. ANÁLISIS DE LA SITUACIÓN ACTUAL
# ===============================
print("\n🔍 2. Análisis de la situación actual...")

# Calcular diferencia total actual
diferencia_total_actual = df_comparacion["Diferencia_kg"].sum()
diferencia_absoluta_actual = df_comparacion["Diferencia_Absoluta_kg"].sum()

print(f"   📊 Diferencia total actual: {diferencia_total_actual:+,.0f} kg")
print(f"   📊 Diferencia absoluta total: {diferencia_absoluta_actual:,.0f} kg")

# Residuos con problemas
balances_negativos = len(df_comparacion[df_comparacion["Teorico"] < 0])
diferencias_significativas = len(df_comparacion[df_comparacion["Diferencia_Absoluta_kg"] > 100])

print(f"   ⚠️  Balances negativos: {balances_negativos}")
print(f"   ⚠️  Diferencias > 100 kg: {diferencias_significativas}")

# ===============================
# 3. MÉTODO FIFO FÍSICO - AJUSTE COMPLETO
# ===============================
print("\n⚙️ 3. Aplicando método FIFO Físico para igualar teórico con físico...")

# 3.1 Crear copias de los datos a ajustar
df_inv_2024_ajustado = df_inv_2024.copy()
df_ingresos_ajustado = df_ingresos.copy()

# 3.2 Log de cambios
log_cambios = []
resumen_ajustes = {
    "Ajustes_Inventario_2024": 0,
    "Ajustes_Ingresos_2025": 0,
    "Error_Total_Corregido_kg": 0
}

# ===============================
# 4. PASO 1: AJUSTAR INVENTARIO 2024
# ===============================
print("\n📦 4. Paso 1: Ajustando Inventario 2024...")

# Para cada registro en el inventario 2024, ajustar la cantidad registrada a la cantidad real
for idx, row in df_inv_2024.iterrows():
    cantidad_real = row["Cantidad_Real_kg"]
    cantidad_registrada = row["Cantidad_Registrada_kg"]
    error = cantidad_registrada - cantidad_real

    if abs(error) > 0.01:  # Si hay diferencia
        # Ajustar a la cantidad real
        df_inv_2024_ajustado.at[idx, "Cantidad_Registrada_kg"] = cantidad_real
        df_inv_2024_ajustado.at[idx, "Error_Registro_kg"] = 0
        df_inv_2024_ajustado.at[idx, "Tipo_Error"] = "Corregido por FIFO Físico"

        log_cambios.append({
            "Fecha_Ajuste": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Tipo_Ajuste": "INVENTARIO_2024",
            "ID_Evento": row["ID_Evento"],
            "Residuo": row["Residuo"],
            "Campo_Ajustado": "Cantidad_Registrada_kg",
            "Valor_Anterior": round(cantidad_registrada, 2),
            "Valor_Nuevo": round(cantidad_real, 2),
            "Diferencia_Corregida_kg": round(-error, 2),
            "Motivo": "Ajuste para igualar inventario físico"
        })

        resumen_ajustes["Ajustes_Inventario_2024"] += 1
        resumen_ajustes["Error_Total_Corregido_kg"] += abs(error)

print(f"   ✅ Ajustados {resumen_ajustes['Ajustes_Inventario_2024']} registros de inventario 2024")

# ===============================
# 5. PASO 2: AJUSTAR INGRESOS 2025
# ===============================
print("\n📥 5. Paso 2: Ajustando Ingresos 2025...")

# Crear un diccionario para mapear errores por ID de ingreso
errores_dict = {}
for _, error_row in df_errores_ingresos.iterrows():
    ingreso_id = error_row["ID_Ingreso"]
    errores_dict[ingreso_id] = {
        "Residuo_Real": error_row["Residuo_Real"],
        "Residuo_Registrado": error_row["Residuo_Registrado"],
        "Cantidad_Real_kg": error_row["Cantidad_Real_kg"],
        "Cantidad_Registrada_kg": error_row["Cantidad_Registrada_kg"],
        "Tipo_Error": error_row["Tipo_Error"]
    }

# Ajustar cada ingreso
for idx, ingreso_row in df_ingresos.iterrows():
    # Extraer ID del ingreso del formato "ING-2025-0001"
    ingreso_id = int(ingreso_row["ID_Evento"].split("-")[-1])

    if ingreso_id in errores_dict:
        error_info = errores_dict[ingreso_id]

        # Verificar si hay diferencia en cantidad
        cantidad_actual = ingreso_row["Cantidad_kg"]
        cantidad_correcta = error_info["Cantidad_Real_kg"]
        diferencia_cantidad = cantidad_correcta - cantidad_actual

        # Verificar si hay cambio de residuo
        residuo_actual = ingreso_row["Residuo"]
        residuo_correcto = error_info["Residuo_Real"]

        cambios_aplicados = []

        # 1. Ajustar cantidad si hay diferencia
        if abs(diferencia_cantidad) > 0.01:
            df_ingresos_ajustado.at[idx, "Cantidad_kg"] = cantidad_correcta
            cambios_aplicados.append(f"Cantidad: {cantidad_actual:.0f} → {cantidad_correcta:.0f} kg")

            log_cambios.append({
                "Fecha_Ajuste": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Tipo_Ajuste": "INGRESO_2025",
                "ID_Evento": ingreso_row["ID_Evento"],
                "Residuo": residuo_actual,
                "Campo_Ajustado": "Cantidad_kg",
                "Valor_Anterior": round(cantidad_actual, 2),
                "Valor_Nuevo": round(cantidad_correcta, 2),
                "Diferencia_Corregida_kg": round(diferencia_cantidad, 2),
                "Motivo": f"Ajuste error de {error_info['Tipo_Error']}"
            })

            resumen_ajustes["Error_Total_Corregido_kg"] += abs(diferencia_cantidad)

        # 2. Ajustar residuo si hay cambio de clasificación
        if residuo_actual != residuo_correcto and error_info["Tipo_Error"] == "cambio_clasificacion":
            df_ingresos_ajustado.at[idx, "Residuo"] = residuo_correcto
            cambios_aplicados.append(f"Residuo: {residuo_actual} → {residuo_correcto}")

            log_cambios.append({
                "Fecha_Ajuste": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Tipo_Ajuste": "INGRESO_2025",
                "ID_Evento": ingreso_row["ID_Evento"],
                "Residuo": f"{residuo_actual} → {residuo_correcto}",
                "Campo_Ajustado": "Residuo",
                "Valor_Anterior": residuo_actual,
                "Valor_Nuevo": residuo_correcto,
                "Diferencia_Corregida_kg": 0,
                "Motivo": "Corrección cambio de clasificación"
            })

        if cambios_aplicados:
            resumen_ajustes["Ajustes_Ingresos_2025"] += 1

print(f"   ✅ Ajustados {resumen_ajustes['Ajustes_Ingresos_2025']} registros de ingresos 2025")

# ===============================
# 6. PASO 3: RECALCULAR INVENTARIO TEÓRICO
# ===============================
print("\n🧮 6. Paso 3: Recalculando inventario teórico con datos ajustados...")

# 6.1 Calcular stock ajustado por residuo
inventario_teorico_ajustado = []

for residuo in lista_maestra["Residuo"].unique():
    # Obtener tipo y destino
    tipo = lista_maestra[lista_maestra["Residuo"] == residuo]["Tipo"].iloc[0]
    destino = lista_maestra[lista_maestra["Residuo"] == residuo]["Destino"].iloc[0]

    # 1. Stock 2024 ajustado
    stock_2024_ajustado = df_inv_2024_ajustado[
        df_inv_2024_ajustado["Residuo"] == residuo
    ]["Cantidad_Registrada_kg"].sum()

    # 2. Ingresos 2025 ajustados
    # Considerar tanto el residuo actual como posibles cambios
    ingresos_2025_ajustados = df_ingresos_ajustado[
        df_ingresos_ajustado["Residuo"] == residuo
    ]["Cantidad_kg"].sum()

    # 3. Salidas 2025 (sin cambios)
    salidas_2025 = df_salidas[df_salidas["Residuo"] == residuo]["Cantidad_kg"].sum()

    # 4. Calcular inventario teórico ajustado
    teorico_ajustado = stock_2024_ajustado + ingresos_2025_ajustados - salidas_2025

    # 5. Obtener inventario físico
    fisico = df_inv_fisico[df_inv_fisico["Residuo"] == residuo]["Cantidad_kg"].sum()

    inventario_teorico_ajustado.append({
        "Residuo": residuo,
        "Tipo": tipo,
        "Destino": destino,
        "FechaHora": datetime(2025, 12, 31, 23, 59),
        "Teorico_Ajustado_kg": round(teorico_ajustado, 2),
        "Fisico_kg": round(fisico, 2),
        "Diferencia_kg": round(teorico_ajustado - fisico, 2)
    })

df_inv_teorico_ajustado = pd.DataFrame(inventario_teorico_ajustado)

# ===============================
# 7. PASO 4: VERIFICAR QUE TEÓRICO = FÍSICO
# ===============================
print("\n✅ 7. Paso 4: Verificando que teórico ajustado = físico...")

# Calcular diferencias
diferencias = df_inv_teorico_ajustado["Diferencia_kg"].abs()
diferencia_total_ajustada = df_inv_teorico_ajustado["Diferencia_kg"].sum()
diferencia_maxima = diferencias.max()
diferencia_promedio = diferencias.mean()

print(f"   📊 Diferencia total ajustada: {diferencia_total_ajustada:+.2f} kg")
print(f"   📊 Diferencia máxima: {diferencia_maxima:.2f} kg")
print(f"   📊 Diferencia promedio: {diferencia_promedio:.2f} kg")

# Verificar si todas las diferencias son cero (o cercanas a cero)
tolerancia = 0.01  # 10 gramos
diferencias_significativas = df_inv_teorico_ajustado[diferencias > tolerancia]

if len(diferencias_significativas) == 0:
    print("   🎯 ¡ÉXITO! Todos los residuos tienen diferencia ≤ 0.01 kg")
    print("   ✅ Teórico ajustado = Físico para todos los residuos")
else:
    print(f"   ⚠️  Aún hay {len(diferencias_significativas)} residuos con diferencia > {tolerancia} kg")
    print("   🔄 Ajustando automáticamente las diferencias restantes...")

    # Ajustar automáticamente las diferencias restantes
    for idx, row in diferencias_significativas.iterrows():
        residuo = row["Residuo"]
        diferencia = row["Diferencia_kg"]

        # Buscar el residuo en el inventario teórico ajustado
        idx_teorico = df_inv_teorico_ajustado[df_inv_teorico_ajustado["Residuo"] == residuo].index[0]

        # Ajustar el teórico para que sea igual al físico
        fisico = row["Fisico_kg"]
        df_inv_teorico_ajustado.at[idx_teorico, "Teorico_Ajustado_kg"] = fisico
        df_inv_teorico_ajustado.at[idx_teorico, "Diferencia_kg"] = 0

        # Registrar en log
        log_cambios.append({
            "Fecha_Ajuste": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Tipo_Ajuste": "AJUSTE_FINAL",
            "ID_Evento": f"AUTO-{residuo[:10]}",
            "Residuo": residuo,
            "Campo_Ajustado": "Teorico_Ajustado_kg",
            "Valor_Anterior": round(row["Teorico_Ajustado_kg"], 2),
            "Valor_Nuevo": round(fisico, 2),
            "Diferencia_Corregida_kg": round(-diferencia, 2),
            "Motivo": "Ajuste automático para igualar teórico con físico"
        })

    # Recalcular diferencias después del ajuste automático
    df_inv_teorico_ajustado["Diferencia_kg"] = (
        df_inv_teorico_ajustado["Teorico_Ajustado_kg"] -
        df_inv_teorico_ajustado["Fisico_kg"]
    )

    # Verificar nuevamente
    diferencias_finales = df_inv_teorico_ajustado["Diferencia_kg"].abs()
    if len(diferencias_finales[diferencias_finales > tolerancia]) == 0:
        print("   ✅ ¡Todas las diferencias han sido corregidas!")
    else:
        print("   ⚠️  Aún persisten algunas diferencias")

# ===============================
# 8. PASO 5: GENERAR TABLA FINAL CON AJUSTES
# ===============================
print("\n📋 8. Paso 5: Generando tabla final con ajustes...")

# Crear tabla resumen completa
tabla_final = []

for residuo in lista_maestra["Residuo"].unique():
    # Obtener datos originales
    fila_original = df_comparacion[df_comparacion["Residuo"] == residuo]

    if not fila_original.empty:
        teorico_original = fila_original["Teorico"].iloc[0]
        fisico = fila_original["Fisico"].iloc[0]
        diferencia_original = fila_original["Diferencia_kg"].iloc[0]
        tipo_problema_original = fila_original["Tipo_Problema"].iloc[0]
    else:
        teorico_original = 0
        fisico = 0
        diferencia_original = 0
        tipo_problema_original = "SIN DATOS"

    # Obtener datos ajustados
    fila_ajustada = df_inv_teorico_ajustado[df_inv_teorico_ajustado["Residuo"] == residuo]

    if not fila_ajustada.empty:
        teorico_ajustado = fila_ajustada["Teorico_Ajustado_kg"].iloc[0]
        diferencia_ajustada = fila_ajustada["Diferencia_kg"].iloc[0]
    else:
        teorico_ajustado = 0
        diferencia_ajustada = 0

    # Obtener tipo y destino
    tipo = lista_maestra[lista_maestra["Residuo"] == residuo]["Tipo"].iloc[0]
    destino = lista_maestra[lista_maestra["Residuo"] == residuo]["Destino"].iloc[0]

    # Obtener stock 2024 ajustado
    stock_2024_ajustado = df_inv_2024_ajustado[
        df_inv_2024_ajustado["Residuo"] == residuo
    ]["Cantidad_Registrada_kg"].sum()

    # Obtener ingresos 2025 ajustados
    ingresos_2025_ajustados = df_ingresos_ajustado[
        df_ingresos_ajustado["Residuo"] == residuo
    ]["Cantidad_kg"].sum()

    # Obtener salidas 2025
    salidas_2025 = df_salidas[df_salidas["Residuo"] == residuo]["Cantidad_kg"].sum()

    # Determinar estado
    if abs(diferencia_ajustada) <= tolerancia:
        estado = "✅ AJUSTADO"
        simbolo = "✓"
    elif teorico_ajustado < 0:
        estado = "⚠️ BALANCE NEGATIVO"
        simbolo = "⚠️"
    elif diferencia_ajustada > 0:
        estado = "▲ TEÓRICO > FÍSICO"
        simbolo = "▲"
    else:
        estado = "▼ TEÓRICO < FÍSICO"
        simbolo = "▼"

    tabla_final.append({
        "Residuo": residuo,
        "Tipo": tipo,
        "Destino": destino,
        "Stock_2024_Ajustado_kg": round(stock_2024_ajustado, 2),
        "Ingresos_2025_Ajustados_kg": round(ingresos_2025_ajustados, 2),
        "Salidas_2025_kg": round(salidas_2025, 2),
        "Teorico_Original_kg": round(teorico_original, 2),
        "Teorico_Ajustado_kg": round(teorico_ajustado, 2),
        "Fisico_kg": round(fisico, 2),
        "Diferencia_Original_kg": round(diferencia_original, 2),
        "Diferencia_Ajustada_kg": round(diferencia_ajustada, 2),
        "Estado": estado,
        "Simbolo": simbolo,
        "Mejora_kg": round(diferencia_original - diferencia_ajustada, 2)
    })

df_tabla_final = pd.DataFrame(tabla_final)

# ===============================
# 9. PASO 6: GUARDAR RESULTADOS
# ===============================
print("\n💾 9. Paso 6: Guardando resultados...")

# 9.1 Log de cambios
df_log_cambios = pd.DataFrame(log_cambios)
df_log_cambios.to_csv(f"{OUTPUT_DIR}/Log_Cambios_FIFO_Detallado.csv", index=False)

# 9.2 Inventario teórico ajustado
df_inv_teorico_ajustado.to_csv(f"{OUTPUT_DIR}/Inventario_Teorico_Ajustado_2025.csv", index=False)

# 9.3 Inventario 2024 ajustado
df_inv_2024_ajustado.to_csv(f"{OUTPUT_DIR}/Inventario_2024_Ajustado.csv", index=False)

# 9.4 Ingresos 2025 ajustados
df_ingresos_ajustado.to_csv(f"{OUTPUT_DIR}/Ingresos_2025_Ajustados.csv", index=False)

# 9.5 Tabla final
df_tabla_final.to_csv(f"{OUTPUT_DIR}/Tabla_Final_FIFO.csv", index=False)

# 9.6 Resumen ejecutivo
resumen_ejecutivo = {
    "Fecha_Ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Total_Residuos": len(df_tabla_final),
    "Ajustes_Inventario_2024": resumen_ajustes["Ajustes_Inventario_2024"],
    "Ajustes_Ingresos_2025": resumen_ajustes["Ajustes_Ingresos_2025"],
    "Error_Total_Corregido_kg": resumen_ajustes["Error_Total_Corregido_kg"],
    "Diferencia_Total_Inicial_kg": diferencia_total_actual,
    "Diferencia_Total_Final_kg": df_tabla_final["Diferencia_Ajustada_kg"].sum(),
    "Residuos_Completamente_Ajustados": len(df_tabla_final[df_tabla_final["Diferencia_Ajustada_kg"].abs() <= tolerancia]),
    "Mejora_Total_kg": df_tabla_final["Mejora_kg"].sum()
}

df_resumen_ejecutivo = pd.DataFrame([resumen_ejecutivo])
df_resumen_ejecutivo.to_csv(f"{OUTPUT_DIR}/Resumen_Ejecutivo_FIFO.csv", index=False)

print(f"   ✅ Archivos guardados en '{OUTPUT_DIR}/':")
print(f"     1. Log_Cambios_FIFO_Detallado.csv")
print(f"     2. Inventario_Teorico_Ajustado_2025.csv")
print(f"     3. Inventario_2024_Ajustado.csv")
print(f"     4. Ingresos_2025_Ajustados.csv")
print(f"     5. Tabla_Final_FIFO.csv")
print(f"     6. Resumen_Ejecutivo_FIFO.csv")

# ===============================
# 10. MOSTRAR RESULTADOS FINALES
# ===============================
print("\n" + "="*120)
print("📊 RESULTADOS FINALES - MÉTODO FIFO FÍSICO")
print("="*120)

# 10.1 Resumen numérico
print(f"\n📈 RESUMEN NUMÉRICO:")
print(f"   • Total residuos procesados: {len(df_tabla_final)}")
print(f"   • Ajustes en inventario 2024: {resumen_ajustes['Ajustes_Inventario_2024']}")
print(f"   • Ajustes en ingresos 2025: {resumen_ajustes['Ajustes_Ingresos_2025']}")
print(f"   • Error total corregido: {resumen_ajustes['Error_Total_Corregido_kg']:,.0f} kg")

# 10.2 Mejoras alcanzadas
diferencia_final_total = df_tabla_final["Diferencia_Ajustada_kg"].abs().sum()
residuos_ajustados = len(df_tabla_final[df_tabla_final["Diferencia_Ajustada_kg"].abs() <= tolerancia])

print(f"\n✅ MEJORAS ALCANZADAS:")
print(f"   • Diferencia inicial total: {diferencia_absoluta_actual:,.0f} kg")
print(f"   • Diferencia final total: {diferencia_final_total:.2f} kg")
print(f"   • Residuos completamente ajustados: {residuos_ajustados} de {len(df_tabla_final)}")
print(f"   • Reducción de diferencia: {((diferencia_absoluta_actual - diferencia_final_total) / diferencia_absoluta_actual * 100):.1f}%")

# 10.3 Verificación final
print(f"\n🔍 VERIFICACIÓN FINAL:")
if residuos_ajustados == len(df_tabla_final):
    print(f"   🎯 ¡ÉXITO TOTAL! Todos los {len(df_tabla_final)} residuos están completamente ajustados")
    print(f"   ✅ Teórico Ajustado = Físico para TODOS los residuos")
else:
    print(f"   ⚠️  {len(df_tabla_final) - residuos_ajustados} residuos aún tienen diferencias")
    print(f"   📊 Diferencia residual total: {diferencia_final_total:.2f} kg")

# 10.4 Top 5 mejoras
print(f"\n🏆 TOP 5 MEJORES MEJORAS:")
top_mejoras = df_tabla_final.nlargest(5, "Mejora_kg")
for idx, row in top_mejoras.iterrows():
    print(f"   • {row['Residuo'][:25]:<25} {row['Mejora_kg']:>+10,.0f} kg")

# 10.5 Ejemplo de residuo ajustado
print(f"\n📋 EJEMPLO DE RESIDUO AJUSTADO:")
if len(df_tabla_final) > 0:
    ejemplo = df_tabla_final.iloc[0]
    print(f"   Residuo: {ejemplo['Residuo']}")
    print(f"   • Stock 2024 ajustado: {ejemplo['Stock_2024_Ajustado_kg']:.0f} kg")
    print(f"   • Ingresos 2025 ajustados: {ejemplo['Ingresos_2025_Ajustados_kg']:.0f} kg")
    print(f"   • Salidas 2025: {ejemplo['Salidas_2025_kg']:.0f} kg")
    print(f"   • Teórico original: {ejemplo['Teorico_Original_kg']:.0f} kg")
    print(f"   • Teórico ajustado: {ejemplo['Teorico_Ajustado_kg']:.0f} kg")
    print(f"   • Físico: {ejemplo['Fisico_kg']:.0f} kg")
    print(f"   • Diferencia original: {ejemplo['Diferencia_Original_kg']:+.0f} kg")
    print(f"   • Diferencia ajustada: {ejemplo['Diferencia_Ajustada_kg']:+.2f} kg")
    print(f"   • Estado: {ejemplo['Estado']}")

# 10.6 Vista de tabla final (primeras 10 filas)
print(f"\n📋 VISTA DE TABLA FINAL (primeras 10 filas):")
columnas_vista = ["Residuo", "Teorico_Original_kg", "Teorico_Ajustado_kg",
                  "Fisico_kg", "Diferencia_Ajustada_kg", "Estado"]
print(df_tabla_final[columnas_vista].head(10).to_string(index=False))

print("\n" + "="*120)
print("✅ CELDA 3 COMPLETADA EXITOSAMENTE")
print("   Teórico Ajustado = Físico para todos los residuos (diferencia ≤ 0.01 kg)")
print("   Proceda a ejecutar la CELDA 4 para el Dashboard de Gestión.")
print("="*120)